# Bygge grafen

Målet er å skape en graf som
- Er fri for skala, og
- som også holder våre kriminelle.
Det vil i praksis si å lese inn grafen, og så legge til nodene (og deres relasjoner) slik vi har forberedt.

For at våre kriminelle skal gli naturlig inn, må de gis relasjoner til de eksisterende på en "naturlig" måte.  Det vil si at hver node må få (et antall) relasjoner tli andre noder som "ligner".  I praksis betyr det å ha større sjande for å knytte seg til noder som alerede har mange kanter.

Nå har (noen) av våre kriminelle trolig flere kanter enn medianen i datasettet, men det blir under støygrensen å kompensere for.

Eneste måten jeg har funnet for å legge inn ny noder på en "normal" måte, er å lage et sett av alle noder, hvor hver node er i settet like mange ganger som det har kanter.  Når én node nå trekkes fra settet vil det sannsynligheten for å trekke en node reflektere nodens sentralitet.





In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import random
import uuid
import scipy
from datetime import datetime, timedelta

For enkelt å kunne tegne (sub)grafer

In [2]:
# Rutine for å tegne en (liten)
def tegne_graf(G, tittel=None):
    node_farge = []
    for n,d in G.nodes(data=True):
        if ":Person" == d["labels"]:
            node_farge += ["red"]
        elif ":Konto" == d["labels"]:
            node_farge += ["skyblue"]
        elif ":Uttak" == d["labels"]:
            node_farge += ["yellow"]
        elif ":Innskudd" == d["labels"]:
            node_farge += ["black"]
        elif ":Transaksjon" == d["labels"]:
           node_farge += ["green"]
        else:
            print(f"Noden: {d}")
            node_farge += ["cyan"]
        #
    #
    pos = nx.spring_layout(G, seed=42)  # Be nodene spre seg utover så godt de kan
    nx.draw(G, with_labels=False, node_color=node_farge, node_size=300)
    plt.title(tittel)
    plt.figure(figsize=(5,5))
#

Neo4j har en velkjent (men ikke støttet?) måte å gi nodene og kantene type (les: *label*) ved innlesing fra `graphml`.  Atributter satt på kanter med navnet `label`(uten s) og atributter satt på noder med navnet `labels`(med s) settes og kan deretter søkes.

In [3]:
import gzip

EG = nx.MultiGraph()
# husk at gzip åpner i 'b'
with gzip.open("data/email.edgelist.txt.gz", "rt") as fd:
    for linje in fd:
         link = linje.split()
         EG.add_edge(int(link[0]), int(link[1]))
    #
#

# Bort med eposter sendt til seg selv
EG.remove_edges_from(nx.selfloop_edges(EG))
isolerte = list(nx.isolates(EG)) # kan ikke bruke iteratorer direkte
EG.remove_nodes_from(isolerte)

# Sette type på nodene, og merke dem unikt
for n in EG.nodes():
    EG.nodes[n]["labels"] = ":Person" #Neo4j-syntax, med s
    EG.nodes[n]["uid"] = str(uuid.uuid4())
# Og kantene
nx.set_edge_attributes(EG, ":Epost", "label") # uten s

print(EG)

MultiGraph with 57189 nodes and 103083 edges


In [14]:
# Hvor mange deler består grafen av.
# Vi bryr oss ikke om retningen for om A->B er B også knyttet til A (he, he, Epstein effekten :-) )

deler = list(nx.connected_components(EG))
print(f"Grafen består av {len(deler)} deler")
# Hent ut antall 
antall = {}
for n, noder in enumerate(deler):
    antall[n] = len(noder)
#
print(f"Største komponent har {antall[0]} noder")
print(f"Nest største har {antall[1]} noder ({deler[1]})")

Grafen består av 185 deler
Største komponent har 56576 noder
Nest største har 5 noder ({18824, 17168, 1395, 18387, 10007})


In [16]:
# Fjerne alle de små
#største = max(nx.connected_components(EG), key=len)
EPOST = EG.subgraph(deler[0]).copy()
print(EPOST)

MultiGraph with 56576 nodes and 102631 edges


Vi skal legge til den "økonomiske kriminaliteten" som vi har laget.  Det må vi gjøre på en "naturlig" måte.  De nye nodene, i tillegg til sine egenskaper, må "gli inn" i landskapet på en naturlig måte.  Vi følger Barabási i [Kapittel 5.2 om *preferential attachment*](https://networksciencebook.com/chapter/5#growth) og knytter våre noder til eksisterende slik at hvilken node vi skal knytte noder til avhenger av nodens sentralitet.

Vi bygger en liste, hvor antall ganger en node er i listen er lik antall kanter noden har.  Det gjør at når vi skal velge en node tilfeldig, er sjansen størst for at vi knytter oss til sentrale noder (i tråd med teorien).

Jeg spør Gemini:
```
Using networkx, without converting the whole graph to a list, how can I find a nrandom node
```
Den svarer
```
random_node = random.sample(G.nodes, 1)[0]
```
Men når jeg fortsetter:
```
Are you sure?  In your code "random_node = random.sample(G.nodes, 1)[0]" seems to create a list before returning one element
```
Svaret er
```
To be intellectually honest: internally, it still performs an $O(n)$ operation, [...]
```
Eller, som alltid: Livet er lettere når man vet svaret.  Forøvrig er det et interessant spørsmål hva *intellectually honest* skal bety (i motsetning til "bare" *honest* mener jeg).

Oh well.

In [25]:
# Bygg listen
alle_noder = []
for u, v in EPOST.edges():
    # Hver kant gir to noder; se kommentar nedenfor
    alle_noder.extend([u,v])
#
print(f"Antall noder i listen: {len(alle_noder)}")

Antall noder i listen: 205262


Antall noder i listen har to noder for hver kant (det er kantene som er driveren, ikke nodene).  Kanten går tross alt fra en node til en annen.  Det viktige er imidlertid at forholdet mellom nodene (i antall) er beholdt.

In [23]:
# Laste inn bakmann
G = nx.read_graphml("grafer/Bakmann.graphml")
Bakmann = nx.MultiGraph(G)
# For å kunne verifisere at vi finner de riktige nodene
nx.set_node_attributes(Bakmann, True, "Bakmann")
# Sjekke at det ser bra ut  
print(Bakmann)

MultiGraph with 110 nodes and 326 edges


In [22]:
# Laste inn "money mule"
G = nx.read_graphml("grafer/Mule15.graphml")
Esel = nx.MultiGraph(G)
# For å kunne verifisere at vi finner de riktige nodene
nx.set_node_attributes(Esel, True, "Esel")
# Verify by checking the number of nodes and edges
print(Esel)


MultiGraph with 64 nodes and 216 edges


In [21]:
# Laste inn deling av utbytte
G = nx.read_graphml("grafer/Utbytte.graphml")
Utbytte = nx.MultiGraph(G)
# For å kunne verifisere at vi finner de riktige nodene
nx.set_node_attributes(Utbytte, True, "Utbytte")
# Verify by checking the number of nodes and edges
print(Utbytte)

MultiGraph with 81 nodes and 159 edges


Da er vi klare til å legge vår kriminalitet til den store grafen

Fordi det "normale" (i den grat noe er normalt i en graf uten skala) så er at noder har 1 relasjon, knytter vi hver av våre nye noder til resten av grafen med 1 kant.  Sannsynligheten er størst for at den knyttes til en sentral node.

In [26]:
import random
print(f"Originalen: {EPOST}")
# Legg de små grafene til i den store
Komplett = EPOST
for g in Bakmann, Esel, Utbytte:
    # Nodene i de små grafene har identifikatorer som er UUID, så ingen er like
    # Vi kan derfor trygt slå dem sammen
    Komplett = nx.union(Komplett, g)
    for n in g:
        tilfeldig = random.choice(alle_noder)
        # Hent en tilfeldig valgt node fra den originale grafen
        Komplett.add_edge(n, tilfeldig, label=":Epost")
#
print(f"Den komplette: {Komplett}")
print(f"Grafen består av {nx.number_connected_components(Komplett)} del")

Originalen: MultiGraph with 56576 nodes and 102631 edges
Den komplette: MultiGraph with 56831 nodes and 103587 edges
Grafen består av 1 del


Vi har nå en stor graf med være kriminelle godt gjemt inne i den.

In [27]:
GRAPHML_FIL = "grafer/Komplett.graphml"

In [28]:
# Lagre grafen
nx.write_graphml(Komplett, GRAPHML_FIL, named_key_ids=True)

Det ser ut som om `networkx` konverterer `True` til `"True"` som ikke er riktig (det skal være `"true"`).  Heller enn å kjempe med det, konverterer vi strengen i filen.

In [29]:
# Filen er liten; les den inn
with open(GRAPHML_FIL, 'r') as fd:
    linjer = fd.read()

linjer = linjer.replace(">True</data>", ">true</data>")

# ut med den igjen
with open(GRAPHML_FIL, 'w') as fd:
    fd.write(linjer)
#

Husk å kopiere filen fra `grafer/`til `neo4j/import`.